# DEMO Jacobian Sensitivity Diagnostics

This notebook rebuilds the DEMO Stage 4 model from persisted public artifacts and runs only `output_sensitivity_analysis` on the full observed grid.

The MAP/Hessian workflow lives in `demo_map_diagnostics.ipynb` so the structural Jacobian diagnostic can be executed, inspected, and cached independently.


In [1]:
# Configuration
WORKSPACE_ID = "DEMO"
SENSITIVITY_N_DRAWS = 16
DIAGNOSTIC_SEED = 42

run_config = {
    "workspace_id": WORKSPACE_ID,
    "sensitivity_n_draws": SENSITIVITY_N_DRAWS,
    "diagnostic_seed": DIAGNOSTIC_SEED,
}


In [2]:
from __future__ import annotations

import json
import logging
import math
import sys
import time
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("causal_ssm_agent.models.ssm_compilation").setLevel(logging.ERROR)
pio.renderers.default = "plotly_mimetype"


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data/DEMO/run/stage-4.json").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


def summary_markdown(mapping: dict[str, Any], title: str) -> Markdown:
    lines = [f"**{title}**", "", "| Field | Value |", "| --- | --- |"]
    for key, value in mapping.items():
        if isinstance(value, (dict, list)):
            text = json.dumps(value, default=str)
        elif isinstance(value, float):
            text = f"{value:.6g}"
        else:
            text = str(value)
        if len(text) > 110:
            text = text[:107] + "..."
        lines.append(f"| `{key}` | {text} |")
    return Markdown("\n".join(lines))


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_PIPELINE = REPO_ROOT / "apps/data-pipeline"
SRC = DATA_PIPELINE / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from causal_ssm_agent.models.ssm.diagnostics import (  # noqa: E402
    get_stage4b_sweep_context,
    output_sensitivity_analysis,
)
from causal_ssm_agent.models.ssm_builder import prepare_model_runtime  # noqa: E402
from causal_ssm_agent.models.ssm_compiler import compile_ssm_artifact  # noqa: E402

run_dir = REPO_ROOT / "data" / WORKSPACE_ID / "run"
stage4 = json.loads((run_dir / "stage-4.json").read_text())
stage1b = json.loads((run_dir / "stage-1b.json").read_text())
data_for_model = pl.read_parquet(run_dir / "stage2-model-data.parquet")

t0 = time.time()
compiled_ssm = compile_ssm_artifact(
    stage4["model_spec"],
    stage4["authored_priors"],
    causal_spec=stage1b["causal_spec"],
)
runtime = prepare_model_runtime(data_for_model, compiled_ssm=compiled_ssm)
sweep_context = get_stage4b_sweep_context(runtime.model)

prep_summary = {
    "repo_root": str(REPO_ROOT),
    "run_dir": str(run_dir),
    "compile_prepare_seconds": round(time.time() - t0, 3),
    "n_rows_long": data_for_model.height,
    "observation_grid_shape": list(runtime.observations.shape),
    "time_grid_length": int(runtime.times.shape[0]),
    "n_manifest": int(runtime.spec.n_manifest),
    "n_latent": int(runtime.spec.n_latent),
    "flat_parameter_dim": int(sweep_context.flat_dim),
    "structural_backend": runtime.inference_structure.structural_backend,
    "resolved_method": runtime.inference_structure.resolved_method,
}

display(summary_markdown(run_config, "Run Configuration"))
display(summary_markdown(prep_summary, "Prepared Runtime"))

**Run Configuration**

| Field | Value |
| --- | --- |
| `workspace_id` | DEMO |
| `sensitivity_n_draws` | 16 |
| `diagnostic_seed` | 42 |

**Prepared Runtime**

| Field | Value |
| --- | --- |
| `repo_root` | /Users/ma9o/Desktop/causal-ssm-agent/trees/main |
| `run_dir` | /Users/ma9o/Desktop/causal-ssm-agent/trees/main/data/DEMO/run |
| `compile_prepare_seconds` | 1.089 |
| `n_rows_long` | 27325 |
| `observation_grid_shape` | [1588, 20] |
| `time_grid_length` | 1588 |
| `n_manifest` | 20 |
| `n_latent` | 6 |
| `flat_parameter_dim` | 59 |
| `structural_backend` | composed |
| `resolved_method` | aux_gibbs |

In [3]:
STATUS_COLORS = {"pass": "#2f855a", "warn": "#d69e2e", "fail": "#c53030", "unknown": "#718096"}
FAMILY_TOKENS = ("lambda", "drift", "diffusion", "obs_r", "obs_shape", "manifest", "t0")
FAMILY_PALETTE = {
    "lambda": "#2b6cb0",
    "drift": "#805ad5",
    "diffusion": "#dd6b20",
    "obs_r": "#319795",
    "obs_shape": "#d69e2e",
    "manifest": "#9f7aea",
    "t0": "#38a169",
    "other": "#718096",
}


def _family_of(parameter: str, interpretable: str) -> str:
    text = f"{parameter} {interpretable}".lower()
    for token in FAMILY_TOKENS:
        if token in text:
            return token
    return "other"


def _status_value(entry: dict[str, Any]) -> str:
    for key in ("normalized_sv_status", "normalized_status", "sv_status", "status"):
        value = entry.get(key)
        if isinstance(value, str):
            return value
    return "unknown"


def _effective_value(entry: dict[str, Any]) -> float | None:
    value = entry.get("normalized_effective_sv", entry.get("normalized_effective_eigenvalue"))
    try:
        out = float(value)
    except (TypeError, ValueError):
        return None
    return out if math.isfinite(out) else None


def _direction_value(direction: dict[str, Any]) -> float:
    raw = direction.get("normalized_singular_value", direction.get("normalized_eigenvalue"))
    try:
        out = float(raw)
    except (TypeError, ValueError):
        return float("nan")
    return out if math.isfinite(out) else float("nan")


def weak_parameter_rows(entries: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for entry in entries:
        status = _status_value(entry)
        if status == "pass":
            continue
        rows.append({
            "parameter": entry.get("parameter"),
            "interpretable_parameter": entry.get("interpretable_parameter"),
            "status": status,
            "normalized_effective": _effective_value(entry),
            "raw_effective": entry.get("effective_sv", entry.get("effective_eigenvalue")),
        })
    return rows


def interpretable_lookup(per_parameter: list[dict[str, Any]]) -> dict[str, str]:
    return {
        str(entry["parameter"]): str(entry.get("interpretable_parameter") or entry["parameter"])
        for entry in per_parameter
    }


def spectrum_figure(values: list[float], title: str, y_title: str) -> go.Figure:
    xs = list(range(1, len(values) + 1))
    ys = [max(abs(float(v)), 1e-12) for v in values]
    fig = go.Figure(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines+markers",
            line={"color": "#2b6cb0", "width": 2},
            marker={"size": 5},
            hovertemplate="rank=%{x}<br>value=%{y:.4g}<extra></extra>",
        )
    )
    fig.add_hline(y=1, line_dash="dash", line_color="#c53030", annotation_text="fail threshold")
    fig.add_hline(y=10, line_dash="dash", line_color="#2f855a", annotation_text="pass threshold")
    fig.update_layout(
        title=title,
        xaxis_title="Spectrum rank",
        yaxis_title=y_title,
        yaxis_type="log",
        template="plotly_white",
        height=400,
        margin={"l": 60, "r": 24, "t": 58, "b": 48},
    )
    return fig


def family_grouped_loading_heatmap(
    *,
    parameter_names: list[str],
    interpretable_names: dict[str, str],
    full_vectors: list[list[float]] | np.ndarray,
    weak_direction_meta: list[dict[str, Any]],
    title: str,
    max_parameters: int = 40,
    loading_threshold: float = 0.05,
) -> go.Figure | None:
    if not weak_direction_meta or not parameter_names:
        return None
    matrix = np.asarray(full_vectors, dtype=float)
    if matrix.ndim != 2 or matrix.shape[0] != len(parameter_names):
        return None
    matrix = np.where(np.isfinite(matrix), matrix, 0.0)

    direction_meta = [d for d in weak_direction_meta if 0 <= int(d["index"]) - 1 < matrix.shape[1]]
    if not direction_meta:
        return None
    direction_indices = [int(d["index"]) - 1 for d in direction_meta]
    sub = matrix[:, direction_indices]
    abs_max = np.max(np.abs(sub), axis=1)
    keep = [i for i in range(len(parameter_names)) if abs_max[i] >= loading_threshold]
    if not keep:
        keep = list(np.argsort(abs_max)[::-1][: min(max_parameters, len(parameter_names))])
    if len(keep) > max_parameters:
        keep = sorted(keep, key=lambda i: -abs_max[i])[:max_parameters]

    annotated = []
    for i in keep:
        name = parameter_names[i]
        family = _family_of(name, interpretable_names.get(name, ""))
        family_rank = FAMILY_TOKENS.index(family) if family in FAMILY_TOKENS else len(FAMILY_TOKENS)
        annotated.append((family_rank, family, name, i))
    annotated.sort(key=lambda row: (row[0], row[2]))
    ordered_indices = [row[3] for row in annotated]
    families_in_order = [row[1] for row in annotated]
    y_labels = [interpretable_names.get(parameter_names[i], parameter_names[i]) for i in ordered_indices]

    z = sub[ordered_indices, :]
    abs_z_max = max(float(np.max(np.abs(z))), 1e-6) if z.size else 1.0
    x_labels = [f"D{d['index']} · {d.get('status', '?')}" for d in direction_meta]
    hover_text = []
    for row_idx, param_idx in enumerate(ordered_indices):
        row_hover = []
        for col_idx, d in enumerate(direction_meta):
            value = _direction_value(d)
            label = interpretable_names.get(parameter_names[param_idx], parameter_names[param_idx])
            row_hover.append(
                f"<b>{label}</b>"
                f"<br>family={families_in_order[row_idx]}"
                f"<br>direction D{d['index']} ({d.get('status', '?')})"
                f"<br>normalized value={value:.3g}"
                f"<br>loading={z[row_idx, col_idx]:+.4f}"
            )
        hover_text.append(row_hover)

    fig = go.Figure(
        go.Heatmap(
            z=z,
            x=x_labels,
            y=y_labels,
            colorscale="RdBu",
            zmin=-abs_z_max,
            zmax=abs_z_max,
            zmid=0,
            text=hover_text,
            hovertemplate="%{text}<extra></extra>",
            colorbar={"title": "signed loading", "thickness": 14},
            xgap=1,
            ygap=1,
        )
    )
    height = min(900, max(360, 22 * len(y_labels) + 140))
    fig.update_layout(
        title=title,
        xaxis_title="Weak direction",
        yaxis_title="Parameter (grouped by family)",
        template="plotly_white",
        height=height,
        margin={"l": 320, "r": 110, "t": 70, "b": 60},
    )
    return fig


def weak_parameter_bar(rows: list[dict[str, Any]], title: str, limit: int = 30) -> go.Figure | None:
    usable = [row for row in rows if row.get("normalized_effective") is not None]
    usable = sorted(usable, key=lambda row: float(row["normalized_effective"]))[:limit]
    if not usable:
        return None
    labels = [row["interpretable_parameter"] or row["parameter"] for row in usable]
    values = [max(abs(float(row["normalized_effective"])), 1e-12) for row in usable]
    colors = [STATUS_COLORS.get(row["status"], STATUS_COLORS["unknown"]) for row in usable]
    hover = [
        f"parameter={row['parameter']}<br>status={row['status']}<br>"
        f"normalized={row['normalized_effective']:.4g}<br>raw={row['raw_effective']}"
        for row in usable
    ]
    fig = go.Figure(
        go.Bar(
            x=values,
            y=labels,
            orientation="h",
            marker_color=colors,
            customdata=hover,
            hovertemplate="%{customdata}<extra></extra>",
        )
    )
    fig.add_vline(x=1, line_dash="dash", line_color="#c53030", annotation_text="fail")
    fig.add_vline(x=10, line_dash="dash", line_color="#2f855a", annotation_text="pass")
    fig.update_layout(
        title=title,
        xaxis_title="Normalized effective value (log scale)",
        xaxis_type="log",
        template="plotly_white",
        height=max(220, 28 * len(usable) + 100),
        margin={"l": 290, "r": 24, "t": 58, "b": 48},
    )
    return fig


## Full-Grid Jacobian Sensitivity

In [4]:
t0 = time.time()
sensitivity = output_sensitivity_analysis(
    runtime.model,
    runtime.times,
    observations=runtime.observations,
    n_draws=SENSITIVITY_N_DRAWS,
    seed=DIAGNOSTIC_SEED,
    sweep_context=sweep_context,
)
sensitivity_payload = {
    "singular_values": sensitivity.singular_values,
    "normalized_singular_values": sensitivity.normalized_singular_values,
    "deficiency_count": sensitivity.deficiency_count,
    "weak_directions": sensitivity.weak_directions,
    "per_parameter": sensitivity.per_parameter,
    "n_draws": sensitivity.n_draws,
    "n_observations": sensitivity.n_observations,
    "n_parameters": sensitivity.n_parameters,
}
sensitivity_weak_parameters = weak_parameter_rows(sensitivity.per_parameter)
sensitivity_summary = {
    "elapsed_seconds": round(time.time() - t0, 3),
    "n_draws_used": sensitivity.n_draws,
    "n_observations": sensitivity.n_observations,
    "n_parameters": sensitivity.n_parameters,
    "deficiency_count": sensitivity.deficiency_count,
    "weak_direction_count": len(sensitivity.weak_directions),
    "weak_parameter_count": len(sensitivity_weak_parameters),
    "smallest_normalized_singular_values": sensitivity.normalized_singular_values[-10:],
}

sensitivity_interpretable = interpretable_lookup(sensitivity.per_parameter)

display(summary_markdown(sensitivity_summary, "Jacobian Sensitivity Summary"))
display(spectrum_figure(
    sensitivity.normalized_singular_values,
    "Jacobian Normalized Singular-Value Spectrum",
    "Normalized singular value",
))

heatmap_fig = family_grouped_loading_heatmap(
    parameter_names=sensitivity.parameter_names,
    interpretable_names=sensitivity_interpretable,
    full_vectors=sensitivity.normalized_right_singular_vectors,
    weak_direction_meta=sensitivity.weak_directions,
    title="Jacobian weak directions: full V matrix grouped by family",
)
if heatmap_fig is not None:
    display(heatmap_fig)

bar_fig = weak_parameter_bar(sensitivity_weak_parameters, "Weak parameters from Jacobian sensitivity")
if bar_fig is not None:
    display(bar_fig)


**Jacobian Sensitivity Summary**

| Field | Value |
| --- | --- |
| `elapsed_seconds` | 21.049 |
| `n_draws_used` | 16 |
| `n_observations` | 441430 |
| `n_parameters` | 59 |
| `deficiency_count` | 3 |
| `weak_direction_count` | 13 |
| `weak_parameter_count` | 21 |
| `smallest_normalized_singular_values` | [7.364866785469925, 6.097428139453937, 4.654769329878789, 3.863858799026602, 2.8561378031397675, 1.75248046... |

## Alternative model: collapse weak latents to known-input proxies

The baseline diagnostic above flags a structural identifiability cluster around `life_events_load` and `social_engagement`: both are observed only through low-rate count manifests with log links, producing a rank-deficient block in the Jacobian (smallest normalized SV ~0.32, several parameters with raw effective SV exactly 0).

This section recompiles the model with two surgical changes and re-runs the same diagnostic:

1. **Anchor swap (cosmetic)** — within `social_engagement`, move the unit-loaded anchor from `social_calendar_event_count` (n≈369, rate≈1) to `imessage_daily_volume` (n≈1588, rate≈8). The spectrum is invariant to this on its own, but it pairs with the next change.
2. **Collapse to known-input proxies** — promote `life_events_load` and `social_engagement` out of `estimation.state_order` and into `estimation.known_inputs`, with source indicators `medical_event_count` and `imessage_daily_volume` respectively. Their other indicators are removed from the measurement model, their dynamics priors (`rho_*`, `sigma_*`) are dropped, and their outbound edges become input→state effects via the SSM's `input_effect` channel. Two edges are dropped: `affective_state → social_engagement` (input cannot be an effect) and `life_events_load → social_engagement` (input→input is meaningless).

**Causal-consistency trade-offs of the collapse:**

- The mood→withdrawal back-edge (`affective_state → social_engagement`) is gone. The data could not identify it in the baseline diagnostic anyway, but its removal means this projection cannot answer "does affect causally reduce engagement?" — only "given observed engagement, what's its effect on affect?".
- `imessage_daily_volume` is a *digital-communication* proxy, not the same theoretical construct as `social_engagement`. Face-to-face contact (`social_calendar_event_count`) and partner breadth (`weekly_unique_partners`) are no longer in the model.
- `life_events_load → social_engagement` (events disrupting the social network) becomes an unmodeled correlation between two exogenous inputs.

These caveats should be reflected in any downstream causal-effect claims.


In [5]:
# Re-run the Jacobian sensitivity under the {anchor-swap + collapse-to-inputs} variant.
import copy

DROP_LATENTS = {"life_events_load", "social_engagement"}
PROXY_FOR = {
    "life_events_load": "medical_event_count",
    "social_engagement": "imessage_daily_volume",
}
INPUT_SCALES = {"medical_event_count": 1.0, "imessage_daily_volume": 8.0}
DROPPED_MANIFESTS = {
    "medical_event_count",
    "acute_stressor_event_count",
    "social_calendar_event_count",
    "imessage_daily_volume",
    "weekly_unique_partners",
}


def _swap_social_anchor(stage1b: dict, stage4: dict) -> tuple[dict, dict]:
    s1b = copy.deepcopy(stage1b)
    s4 = copy.deepcopy(stage4)
    indicators = s1b["causal_spec"]["measurement"]["indicators"]
    se_block = [ind for ind in indicators if ind.get("construct_name") == "social_engagement"]
    se_imessage = [ind for ind in se_block if ind["name"] == "imessage_daily_volume"]
    se_rest = [ind for ind in se_block if ind["name"] != "imessage_daily_volume"]
    se_reordered = se_imessage + se_rest
    new_indicators: list[dict] = []
    inserted = False
    for ind in indicators:
        if ind.get("construct_name") == "social_engagement":
            if not inserted:
                new_indicators.extend(se_reordered)
                inserted = True
        else:
            new_indicators.append(ind)
    s1b["causal_spec"]["measurement"]["indicators"] = new_indicators

    OLD = "lambda_imessage_daily_volume_social_engagement"
    NEW = "lambda_social_calendar_event_count_social_engagement"
    priors = s4["authored_priors"]
    removed = priors.pop(OLD)
    priors[NEW] = {**removed, "parameter": NEW, "reasoning": "(experiment) anchor swap"}
    for p in s4["model_spec"]["parameters"]:
        if p.get("name") == OLD:
            p["name"] = NEW
            p["description"] = "Factor loading for social_calendar_event_count on social_engagement"
    for rp in s4.get("resolved_priors", []):
        if isinstance(rp, dict) and rp.get("parameter") == OLD:
            rp["parameter"] = NEW
    return s1b, s4


def _is_dropped_param(name: str) -> bool:
    if name in {
        "rho_life_events_load",
        "rho_social_engagement",
        "sigma_life_events_load",
        "sigma_social_engagement",
    }:
        return True
    if name.startswith("manifest_mean_") and name.removeprefix("manifest_mean_") in DROPPED_MANIFESTS:
        return True
    if name.startswith("obs_sd_") and name.removeprefix("obs_sd_") in DROPPED_MANIFESTS:
        return True
    if name.startswith("lambda_"):
        tail = name.removeprefix("lambda_")
        if any(tail.endswith(f"_{latent}") for latent in DROP_LATENTS):
            return True
    if name.startswith("beta_"):
        tail = name.removeprefix("beta_")
        if any(tail.endswith(f"_{latent}") for latent in DROP_LATENTS):
            return True
    return False


def _collapse_latents_to_inputs(stage1b: dict, stage4: dict) -> tuple[dict, dict]:
    s1b = copy.deepcopy(stage1b)
    s4 = copy.deepcopy(stage4)
    cs = s1b["causal_spec"]
    est = cs["estimation"]
    latent = cs["latent"]
    measurement = cs["measurement"]

    est["state_order"] = [s for s in est["state_order"] if s not in DROP_LATENTS]

    existing_inputs = {ki["construct"] for ki in est.get("known_inputs", [])}
    for name in ("life_events_load", "social_engagement"):
        if name in existing_inputs:
            continue
        est["known_inputs"].append({
            "construct": name,
            "source_indicator": PROXY_FOR[name],
            "scale": INPUT_SCALES[PROXY_FOR[name]],
            "missing_policy": "zero" if name == "life_events_load" else "forward_fill",
        })

    est["edges"] = [
        e for e in est.get("edges", [])
        if e["effect"] not in DROP_LATENTS
        and not (e["cause"] in DROP_LATENTS and e["effect"] in DROP_LATENTS)
    ]
    latent["edges"] = [
        e for e in latent.get("edges", [])
        if e["effect"] not in DROP_LATENTS
        and not (e["cause"] in DROP_LATENTS and e["effect"] in DROP_LATENTS)
    ]
    measurement["indicators"] = [
        ind for ind in measurement["indicators"]
        if ind.get("construct_name") not in DROP_LATENTS
    ]

    s4["model_spec"]["likelihoods"] = [
        lk for lk in s4["model_spec"]["likelihoods"]
        if lk.get("variable") not in DROPPED_MANIFESTS
    ]
    s4["model_spec"]["parameters"] = [
        p for p in s4["model_spec"]["parameters"]
        if not _is_dropped_param(p.get("name", ""))
    ]
    s4["authored_priors"] = {k: v for k, v in s4["authored_priors"].items() if not _is_dropped_param(k)}
    s4["resolved_priors"] = [
        rp for rp in s4.get("resolved_priors", [])
        if not (isinstance(rp, dict) and _is_dropped_param(rp.get("parameter", "")))
    ]
    return s1b, s4


# Apply patches in memory and recompile.
stage1b_collapsed, stage4_collapsed = _swap_social_anchor(stage1b, stage4)
stage1b_collapsed, stage4_collapsed = _collapse_latents_to_inputs(
    stage1b_collapsed, stage4_collapsed
)

t0_alt = time.time()
compiled_alt = compile_ssm_artifact(
    stage4_collapsed["model_spec"],
    stage4_collapsed["authored_priors"],
    causal_spec=stage1b_collapsed["causal_spec"],
)
runtime_alt = prepare_model_runtime(data_for_model, compiled_ssm=compiled_alt)
sweep_alt = get_stage4b_sweep_context(runtime_alt.model)

sensitivity_alt = output_sensitivity_analysis(
    runtime_alt.model,
    runtime_alt.times,
    observations=runtime_alt.observations,
    n_draws=SENSITIVITY_N_DRAWS,
    seed=DIAGNOSTIC_SEED,
    sweep_context=sweep_alt,
)
elapsed_alt = round(time.time() - t0_alt, 3)

weak_params_alt = weak_parameter_rows(sensitivity_alt.per_parameter)
summary_alt = {
    "elapsed_seconds": elapsed_alt,
    "n_parameters": sensitivity_alt.n_parameters,
    "n_observations": sensitivity_alt.n_observations,
    "deficiency_count": sensitivity_alt.deficiency_count,
    "weak_direction_count": len(sensitivity_alt.weak_directions),
    "weak_parameter_count": len(weak_params_alt),
    "smallest_normalized_singular_values": sensitivity_alt.normalized_singular_values[-10:],
}

# Side-by-side comparison vs the baseline `sensitivity` from the cell above.
comparison_rows = [
    ("n_parameters", sensitivity.n_parameters, sensitivity_alt.n_parameters),
    ("deficiency_count", sensitivity.deficiency_count, sensitivity_alt.deficiency_count),
    ("weak_direction_count", len(sensitivity.weak_directions), len(sensitivity_alt.weak_directions)),
    ("weak_parameter_count", len(sensitivity_weak_parameters), len(weak_params_alt)),
    (
        "smallest_normalized_sv",
        round(min(sensitivity.normalized_singular_values), 4),
        round(min(sensitivity_alt.normalized_singular_values), 4),
    ),
]
comparison_md_lines = [
    "**Baseline vs. collapsed-proxy variant**",
    "",
    "| metric | baseline | collapsed | delta |",
    "| --- | ---: | ---: | --- |",
]
for name, base_val, alt_val in comparison_rows:
    try:
        delta = float(alt_val) - float(base_val)
        arrow = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
        delta_text = f"{arrow} {delta:+g}"
    except (TypeError, ValueError):
        delta_text = "n/a"
    comparison_md_lines.append(f"| `{name}` | {base_val} | {alt_val} | {delta_text} |")
display(Markdown("\n".join(comparison_md_lines)))

display(summary_markdown(summary_alt, "Collapsed-Proxy Variant Summary"))
display(spectrum_figure(
    sensitivity_alt.normalized_singular_values,
    "Jacobian Normalized Singular-Value Spectrum (collapsed proxy variant)",
    "Normalized singular value",
))
bar_alt = weak_parameter_bar(weak_params_alt, "Remaining weak parameters under collapsed variant")
if bar_alt is not None:
    display(bar_alt)


**Baseline vs. collapsed-proxy variant**

| metric | baseline | collapsed | delta |
| --- | ---: | ---: | --- |
| `n_parameters` | 59 | 45 | ↓ -14 |
| `deficiency_count` | 3 | 0 | ↓ -3 |
| `weak_direction_count` | 13 | 5 | ↓ -8 |
| `weak_parameter_count` | 21 | 6 | ↓ -15 |
| `smallest_normalized_sv` | 0.3173 | 1.4669 | ↑ +1.1496 |

**Collapsed-Proxy Variant Summary**

| Field | Value |
| --- | --- |
| `elapsed_seconds` | 12.258 |
| `n_parameters` | 45 |
| `n_observations` | 304294 |
| `deficiency_count` | 0 |
| `weak_direction_count` | 5 |
| `weak_parameter_count` | 6 |
| `smallest_normalized_singular_values` | [25.20521708461056, 21.601767883494226, 20.214905196097344, 15.61943751432948, 10.214194176992788, 9.830757... |

### Causal-graph after the collapse

```
States (4):   affective_state, sleep_quality, physical_activity, cyp2c19_metabolizer_status
Inputs:       serotonergic_exposure, seasonal_load, prescription_event, adherence,
              life_events_load   (proxy = medical_event_count)
              social_engagement  (proxy = imessage_daily_volume)

Retained edges (input or state → state):
  serotonergic_exposure  → affective_state, physical_activity, sleep_quality
  physical_activity      → affective_state
  sleep_quality          → affective_state
  social_engagement (in) → affective_state          [previously bidirectional]
  affective_state        → physical_activity, sleep_quality
  seasonal_load          → affective_state
  life_events_load (in)  → affective_state

Dropped:
  affective_state         → social_engagement        [back-edge of feedback loop]
  life_events_load        → social_engagement        [input → input]
```

**Interpretation:** the collapsed projection trades two unidentifiable causal claims for a well-conditioned Jacobian. The data couldn't have identified those edges in the baseline anyway (they sat in the rank-deficient block of the Jacobian), so what's lost is a *theoretical* commitment, not an empirical one. Any downstream causal effect estimate from this variant should be framed in terms of the surviving edges only.
